# Predicting Diabetes Progression from Baseline Measurements

**Sultan Akinola Atanda** · Student ID 9114837 · CSCN8010 Foundations of Machine Learning · Conestoga College

The goal is a model that estimates how far a patient's diabetes will progress one year after a baseline visit, so a physician can use it as a first-pass screening tool. Four model families are compared: univariate polynomial regression, multivariate polynomial regression, decision trees and k-nearest neighbours. Logistic regression is added as a direct high-risk classifier.

**Contents**

1. [Getting the data](#1-getting-the-data)
2. [Framing the problem](#2-framing-the-problem)
3. [Exploratory data analysis](#3-exploratory-data-analysis)
4. [Cleaning](#4-cleaning)
5. [Train, validation and test split](#5-train-validation-and-test-split)
6. [Part 2: univariate polynomial models on BMI](#part-2-univariate-polynomial-models-on-bmi)
7. [Part 3: multivariate models](#part-3-multivariate-models)
8. [Overall conclusions](#overall-conclusions)
9. [Screening dashboard](#screening-dashboard)

The heavy lifting lives in three helper modules under `src/`, so the notebook reads as an analysis rather than a wall of code:

| Module | Class(es) | Responsibility |
|---|---|---|
| `data_pipeline.py` | `DiabetesData` | loading, renaming, summaries, quality checks, splitting |
| `visuals.py` | `DiabetesVisualizer` | every Plotly chart and the PNG export used by the README |
| `modeling.py` | `ModelFactory`, `RegressionEvaluator`, `ClassificationEvaluator`, `ModelInspector`, `ModelRegistry` | building, scoring, inspecting and saving models |

## 1. Getting the data

`load_diabetes()` returns the features mean-centred and scaled by default. I load them with `scaled=False` instead so every column stays in its clinical unit (BMI in kg/m², blood pressure in mm Hg). That keeps the BMI equation in Part 2 readable and lets the dashboard accept the numbers a clinician would actually type. Any scaling a model needs happens inside its own pipeline, fitted on the training set only.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from src.data_pipeline import DiabetesData
from src.modeling import (
    ClassificationEvaluator,
    ModelFactory,
    ModelInspector,
    ModelRegistry,
    RegressionEvaluator,
)
from src.visuals import DiabetesVisualizer

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.precision", 3)

data = DiabetesData(random_state=42)
df = data.load()
viz = DiabetesVisualizer(image_dir="images", export=True)

print(f"{df.shape[0]} patients, {df.shape[1] - 1} features")
df.head()

The six blood serum columns arrive as `s1` to `s6`. The dataset description gives their meaning, so I renamed them to something a reader can follow:

In [ ]:
data.dictionary()

## 2. Framing the problem

**What is being predicted.** The target is a quantitative measure of disease progression one year after baseline, ranging from 25 to 346. It is continuous, so the core task is **supervised regression** on 442 patients and 10 baseline features (age, sex, BMI, blood pressure and six serum measurements). The data comes from the study used in Efron et al. (2004).

**How the model would be used.** A screening tool does not need to be exact for every patient. It needs to reliably flag the patients who are heading for a bad outcome so they get a closer look. That makes the errors asymmetric:

- A **false negative** (a high-risk patient told they are fine) can delay treatment. This is the expensive mistake.
- A **false positive** (a stable patient flagged for follow-up) costs an extra appointment.

**Talking points carried over from the workshops**

- *Performance Metrics Classification workshop.* Accuracy hides failure on the minority class. With imbalanced outcomes, precision, recall and the confusion matrix tell the real story, and for screening, **recall on the high-risk group** is the number to protect. I define "high risk" as a score in the top quarter of the training targets and use that cut-off to judge every model on whether it would actually catch these patients, not only on average error.
- *K-Nearest Neighbors workshop.* kNN predicts by averaging the closest training patients, so it depends entirely on distance. Features measured in different units (cholesterol in the hundreds, the triglyceride log near 4.6) must be standardised first or the large-unit columns dominate. Small *k* follows noise, large *k* oversmooths. kNN also learns no weights at all, which matters for the parameter count in step 13.
- *Logistic Regression Classifier workshop.* Logistic regression outputs a probability through the sigmoid and is trained by minimising **log-loss** (cross-entropy), which punishes confident wrong answers far more than hesitant ones. It is a classifier, so it needs a yes/no target. I build that target from the same high-risk cut-off, which turns the question into "what is the chance this patient is high risk?". For a screening tool this is arguably the more natural framing.

**Metrics for the regression models**

| Metric | What it tells us | What to watch for |
|---|---|---|
| R² | share of the variance in progression the model explains | can be negative on new data when a model is worse than predicting the mean |
| MAE | average miss in progression points | same unit as the target, easy to explain to a clinician |
| MAPE | average miss relative to the true value | inflates for patients with small scores (a miss of 25 on a true value of 25 is 100%) |

**Baseline.** A degree-0 polynomial predicts the training mean for everyone. Any model worth using has to beat it clearly.

## 3. Exploratory data analysis

### 3.1 Summary statistics

In [ ]:
data.summary()

- Every column has 442 values and the ranges are physically sensible: ages 19 to 79, BMI 18.0 to 42.2, blood pressure 62 to 133 mm Hg.
- The target is the most variable column by far (coefficient of variation 51%), with a mean of 152 and a median of 140.5. The gap between them points to a right tail of fast progressors.
- `sex` is binary, coded 1 and 2. The source does not say which code is which, so I treat it as a group label only.
- `hdl` (skew 0.80) and `tch` (0.74) are the most skewed features. `ltg` is already a log, which is why it looks so well-behaved.

### 3.2 Data quality

In [ ]:
print(f"Duplicate rows: {data.duplicate_count()}")
data.quality_report()

No missing values and no duplicates. `tch` has only 66 distinct values across 442 patients because the ratio was recorded at low precision, so it behaves almost like an ordinal variable. The IQR rule flags a handful of points in most columns. I look at whether they are believable in section 4.

### 3.3 The target

In [ ]:
data.split()
threshold = data.risk_threshold(0.75)
viz.target_distribution(df[data.TARGET], threshold=threshold)

The distribution is broad and lopsided: a dense block between roughly 50 and 150, then a long, thin tail to 346. The red line marks the high-risk cut-off used later (the 75th percentile of the training targets). Fewer patients sit in the tail, so any model trained to minimise average error will be pulled toward the middle. I expect the models to under-predict exactly the patients a screening tool cares about most.

### 3.4 Feature distributions

In [ ]:
viz.histograms(df, data.features)

- `bmi`, `bp` and `glu` are close to bell-shaped with a mild right lean.
- `hdl` has a visible right tail (a few patients near 99 mg/dL).
- `tch` shows the stepped, spiky shape that comes from coarse recording.
- `sex` is split 235 / 207, which is balanced enough that no group is under-represented.

### 3.5 Features against progression

In [ ]:
viz.scatter_against_target(df, data.features, data.TARGET)

In [ ]:
data.target_correlations().to_frame('r with progression')

- **BMI (r = 0.59)** and **ltg (r = 0.57)** are the strongest single signals, followed by blood pressure (0.44) and `tch` (0.43).
- `hdl` is the only clearly *negative* relationship (-0.39): higher "good" cholesterol goes with slower progression, which agrees with clinical intuition.
- Even the best scatter plot is a wide cloud. At any given BMI, progression spans more than 200 points, so a single feature will leave most of the variance unexplained. That sets realistic expectations for Part 2.
- `sex` (0.04), `ldl` (0.17), `age` (0.19) and `tc` (0.21) carry little linear signal on their own.

### 3.6 Correlation matrix

In [ ]:
viz.correlation_heatmap(df)

- `tc` and `ldl` are almost the same measurement (r = 0.90). LDL makes up most of total cholesterol, so this is expected.
- `hdl` and `tch` are strongly opposed (r = -0.74), which follows from `tch` being total cholesterol *divided by* HDL.
- `tch` also correlates with `ldl` (0.66) and `ltg` (0.62). The cholesterol block is heavily redundant.
- This redundancy is harmless for trees but it inflates the coefficients of polynomial models and gives kNN duplicated dimensions that count twice in the distance. I use this in Part 3 to decide which features to drop.

### 3.7 Outliers

In [ ]:
viz.standardized_boxplots(df, data.features)

On a common z-score scale, the extreme points sit in `hdl`, `tch`, `glu` and `tc`, and none is more than about four standard deviations out. These are unusual patients, not impossible ones.

**EDA takeaways**

1. BMI is the best single predictor, which justifies its role in Part 2, but it explains only about a third of the variance on its own.
2. `ltg`, `bp`, `tch` and `hdl` add separate information, so the multivariate models in Part 3 should do clearly better.
3. The cholesterol features overlap heavily. `ldl` and `tc` are the obvious candidates to remove.
4. The target has a long upper tail. Models will tend to under-predict the fast progressors, so error has to be checked by range, not only on average.

## 4. Cleaning

The dataset arrives clean, so this step is mostly about deciding what *not* to change:

- **Missing values and duplicates:** none, so no imputation or removal.
- **Column names:** `s1` to `s6` were renamed to their clinical meaning (done in `DiabetesData.load`).
- **Types:** `sex` was cast to an integer since it only takes two values.
- **Outliers:** kept. Below are the patients flagged on BMI, the feature Part 2 depends on.

In [ ]:
data.outlier_rows('bmi')

A BMI of 42 is severe obesity, not a typo, and these patients have high progression scores, which is exactly the group a screening tool exists for. Removing them would make the test metrics look better while making the tool worse at its job. The same reasoning applies to the other flagged values, which all fall inside normal clinical ranges.

- **Scaling:** deliberately not done here. Scaling before splitting would leak the test set's mean and spread into training. Each model that needs it scales inside its own pipeline.

## 5. Train, validation and test split

75% train, 10% validation and 15% test, with a fixed seed so every run gives the same split.

In [ ]:
splits = data.split(train=0.75, val=0.10, test=0.15)
data.split_overview()

The validation set has only 44 patients and its mean target (131) is noticeably lower than the training mean (154). With so few rows, one or two unusual patients can swing a validation score. Where the validation ranking is close or surprising, I double-check it with 10-fold cross-validation on the training set, which never touches the test data.

---

# Part 2: univariate polynomial models on BMI

## 6. Six polynomial models, degree 0 to 5

Each model is a pipeline: standardise BMI, expand it into polynomial terms, then fit ordinary least squares. Raw BMI to the fifth power reaches about 1.3 × 10⁸, which makes the least squares problem numerically unstable. Fitting on standardised BMI avoids that, and step 11 converts the coefficients back to raw BMI.

In [ ]:
bmi_splits = data.subset(["bmi"])
part2 = RegressionEvaluator(bmi_splits)
part2.fit({f"degree_{d}": ModelFactory.univariate_polynomial(d) for d in range(6)})

X_train_bmi, y_train = bmi_splits["train"]
viz.polynomial_family(part2.models, X_train_bmi, y_train)

Degree 0 is the flat mean line. Degrees 1 to 3 are almost indistinguishable inside the bulk of the data. Degrees 4 and 5 start to bend at the edges, where there are only a few patients to pin the curve down. The extra flexibility is being spent on the sparse ends of the BMI range.

## 7. Training and validation comparison

In [ ]:
part2_table = part2.comparison_table()
part2_table

In [ ]:
viz.metric_comparison(part2_table, "R2", title="BMI models: R² on training vs validation", name="part2_r2")

In [ ]:
viz.metric_comparison(part2_table, "MAE", title="BMI models: MAE on training vs validation", name="part2_mae")

- Degree 0 explains nothing (R² = 0 on train, -0.11 on validation), as a mean predictor should.
- Moving to degree 1 cuts validation MAE from 66 to 47 points. That single step captures almost all the value BMI has to offer.
- Degrees 1 to 3 are tied to within 0.05 MAE points. Degrees 4 and 5 look a little better on validation (45.5 MAE, R² 0.35 for degree 5).
- The train-validation R² gap stays under 0.02 for every non-trivial degree, so none of these models is badly overfitting in the usual sense.
- Validation MAE is *lower* than training MAE across the board. That is the small, easier validation set at work, not a sign the models generalise better than they fit.

## 8. Choosing the best model

On the table alone, **degree 5** wins. Before trusting a gain of under two MAE points on 44 patients, I check whether it holds up on other slices of the training data:

In [ ]:
part2.cross_validation_table()

In [ ]:
grid = pd.DataFrame({"bmi": [16, 18, 20, 30, 40, 42, 45]})
grid.assign(
    degree_1=part2.models["degree_1"].predict(grid).round(1),
    degree_5=part2.models["degree_5"].predict(grid).round(1),
)

The validation win does not survive scrutiny:

- Across 10 folds, degree 1 has the lowest average MAE (52.7, tied with degree 2) and degree 5 is the worst of the non-trivial models (53.2). The validation advantage was luck of the split.
- Degree 5 is not monotonic. It predicts *higher* progression for a BMI of 16 (181) than for 20 (90), and progression starts to *fall* above BMI 42. No clinician would accept a screening rule that rates a very underweight patient as higher risk than a normal-weight one.

**Chosen model: degree 1.** It ties for the best cross-validated error, it is the simplest, and its behaviour is monotonic and explainable.

In [ ]:
best_bmi = "degree_1"
bmi_model = part2.models[best_bmi]
part2.test_report(best_bmi)

## 9. Test set results

| Split | R² | MAE | MAPE |
|---|---|---|---|
| Test | 0.281 | 51.5 | 43.2% |

On unseen patients the model explains about 28% of the variance and misses by about 52 progression points on average. The test R² is below the training R² (0.350), a moderate drop that is consistent with the cross-validated estimate (0.289). The model is not overfitting; it is simply limited by having one input.

## 10. The fit over train, validation and test

In [ ]:
viz.fit_over_splits(bmi_model, bmi_splits)

The line follows the general upward trend in all three sets, but the vertical scatter around it is large and similar in every panel. The same pattern shows up in each split: points above about 250 sit well above the line and points below about 75 sit well below it.

## 11. Equation of the chosen model

In [ ]:
print(ModelInspector.polynomial_equation(bmi_model, variable='BMI'))

$$\text{progression} = -125.18 + 10.51 \times \text{BMI}$$

Each extra unit of BMI adds about **10.5 progression points**. Two decimals are enough for this model: rounding the coefficients changes a prediction by well under one point. The intercept has no physical meaning on its own (nobody has a BMI of 0); it only positions the line. For the degree 4 and 5 models two decimals would *not* be sufficient, because their higher-order coefficients round to 0.00 while still mattering after being multiplied by BMI⁵.

## 12. Prediction for a chosen BMI

In [ ]:
example_bmi = 32.0
expected = bmi_model.predict(pd.DataFrame({"bmi": [example_bmi]}))[0]
print(f"Expected progression at BMI {example_bmi}: {expected:.1f}")
print(f"High-risk cut-off: {threshold:.0f}")

A patient with a BMI of 32 (obese class I) is expected to reach about **211**, sitting just below the high-risk line of 214. On BMI alone, a clearly obese patient is still not flagged. That is the first hint of the screening problem discussed in the conclusions.

## 13. Trainable parameters

In [ ]:
pd.DataFrame(
    {
        "terms (get_feature_names_out)": [
            ", ".join(model.named_steps["poly"].get_feature_names_out()) for model in part2.models.values()
        ],
        "trainable parameters": [ModelInspector.parameter_count(m) for m in part2.models.values()],
    },
    index=part2.models.keys(),
)

A degree-*d* model on one feature fits **d + 1** coefficients: one per power of BMI from 0 to *d*. `PolynomialFeatures(include_bias=True)` creates the column of ones (shown as `1`) that carries the intercept, so the regression is fitted with `fit_intercept=False` and the intercept is not counted twice. Degree 0 fits a single number (the mean); degree 5 fits six.

The `StandardScaler` also estimates a mean and a standard deviation from the training data, but those are computed directly rather than learned by minimising the error, so they are not counted as trainable.

## 14. Part 2 conclusions

**Performance.** The degree-1 BMI model is a real improvement on guessing the mean (test MAE 51.5 vs 66 for the baseline), but with a test R² of 0.28 it leaves about 72% of the variation in progression unexplained.

**Where the model fails**

In [ ]:
bmi_test_X, bmi_test_y = bmi_splits["test"]
viz.actual_vs_predicted(
    bmi_test_y, bmi_model.predict(bmi_test_X), "BMI model on the test set: actual vs predicted",
    "part2_actual_vs_predicted", threshold=threshold,
)

In [ ]:
part2_bands = part2.error_by_range(best_bmi, split="test")
viz.error_by_band(part2_bands, "BMI model errors by actual progression (test set)", "part2_error_bands")
part2_bands

In [ ]:
part2.screening_table(threshold, names=[best_bmi], split='test')

- **It compresses everything toward the middle.** Predictions only move from about 64 to 318 as BMI goes from its lowest to its highest test value, yet patients with the same BMI differ by well over 150 points, and the points form a flat band rather than following the diagonal.
- **It over-predicts the slowest progressors** (mean bias +51 in the bottom quarter, MAPE 86%). This is where MAPE explodes, because the true values are small.
- **It badly under-predicts the fastest progressors** (mean bias -66 in the top quarter, MAE 75). These are the patients that matter most.
- **As a screener it misses most high-risk patients.** Using the 214 cut-off, it catches only 5 of 19 high-risk test patients (recall 0.26). The few it does flag are mostly correct (precision 0.71), but in screening a confident miss is worse than a false alarm.
- **By BMI band**, the largest errors are in the overweight range (25 to 29), where patients with similar BMI have very different outcomes.

**Limitations**

- One feature cannot separate patients who look identical on BMI but differ in blood pressure or triglycerides.
- A straight line assumes the effect of BMI is the same at every level and for every patient.
- 442 patients from a single study. The tool would need validating on the clinic's own population before use.
- MAPE overstates the error for low-progression patients, so MAE is the more trustworthy summary here.
- The model describes an association. It does not show that lowering BMI would change a given patient's progression.

**Workshop connections.** The screening table is the Performance Metrics workshop in action: a model with respectable average error can still have poor recall on the class that matters. The logistic regression workshop points to the fix, which is to model the high-risk outcome directly as a probability. Part 3 tests both ideas.

---

# Part 3: multivariate models

## 15. Feature selection

Based on the EDA, I drop `ldl` and `tc`:

- They are near duplicates of each other (r = 0.90) and are the two weakest cholesterol signals for the target (0.17 and 0.21).
- Their information largely survives in `tch` (the total-cholesterol-to-HDL ratio) and `hdl`.
- Removing them shortens kNN distances by two redundant dimensions and cuts the degree-3 polynomial from 286 to 165 coefficients.

I keep `sex` despite its weak correlation. A near-zero correlation means no *straight-line* relationship on its own, but it can still matter in interaction with other features, which the trees and polynomials can pick up.

In [ ]:
selected = [f for f in data.features if f not in ("ldl", "tc")]
multi_splits = data.subset(selected)
print(selected)

## 16. Models

| Model | Setting | Why this pair |
|---|---|---|
| Polynomial | degree 2, degree 3 | adds squares and pairwise interactions, then three-way terms |
| Decision tree | `max_depth=3`, `max_depth=5` with `min_samples_leaf=5` | a shallow, readable tree vs a deeper one held back by leaf size |
| kNN | `k=5`, `k=11` | a local, low-bias version vs a smoother one |
| Linear (reference) | degree 1 | an honest baseline to judge whether the extra complexity pays off |

The polynomial and kNN pipelines standardise features inside the pipeline. Trees split on one feature at a time, so they do not need scaling.

In [ ]:
part3 = RegressionEvaluator(multi_splits)
part3.fit(
    {
        "poly_degree_2": ModelFactory.multivariate_polynomial(2),
        "poly_degree_3": ModelFactory.multivariate_polynomial(3),
        "tree_depth_3": ModelFactory.decision_tree(max_depth=3),
        "tree_depth_5": ModelFactory.decision_tree(max_depth=5, min_samples_leaf=5),
        "knn_k5": ModelFactory.knn(5),
        "knn_k11": ModelFactory.knn(11),
        "linear_reference": ModelFactory.multivariate_polynomial(1),
    }
)
part3_table = part3.comparison_table()
part3_table

In [ ]:
viz.leaderboard(part3_table, 'MAE')

In [ ]:
part3.cross_validation_table()

## 17. Comparison

- **Degree-3 polynomial overfits dramatically.** Training R² 0.71, validation R² -0.14: worse than predicting the mean. With 165 coefficients and only 331 training rows, it memorises noise. Its cross-validated MAE (84) confirms this is not a fluke of the split.
- **Degree-2 polynomial** is solid (validation R² 0.40, best cross-validated MAE of the required models at 48.3) but still shows a train-validation gap of 0.17.
- **Trees** fit training data well and generalise poorly. The depth-5 tree has a 0.34 R² gap, and even the depth-3 tree only reaches 0.20 validation R². Hard thresholds make a step-shaped prediction for what is really a gradual relationship.
- **kNN with k = 11** posts the best validation R² (0.45), close to the best validation MAE (40.9), and the smallest train-validation gap of any model (0.05). Moving from k = 5 to k = 11 trades a little training fit for noticeably better stability.
- **The plain linear reference** is a strong performer. It has the lowest cross-validated MAE (46.0) of everything tested.

## 18. Best model

Among the required model families I choose **kNN with k = 11**. It has the best validation R², a validation MAE within 0.3% of the leader, the smallest generalisation gap, and a cross-validated MAE within one standard deviation of the best. kNN with k = 5 has a slightly lower validation MAE but a larger gap and a worse cross-validated score.

The linear reference model is worth stating plainly: it does about as well as kNN on validation and better in cross-validation, while being far simpler. I come back to this in the conclusions.

In [ ]:
best_multi = "knn_k11"
knn_model = part3.models[best_multi]
part3.test_report(best_multi)

## 19. Test set results

| Split | R² | MAE | MAPE |
|---|---|---|---|
| Test | 0.490 | 42.8 | 36.3% |

Compared with the BMI-only model, test R² nearly doubles (0.28 to 0.49) and MAE drops by about 9 points. Using several features clearly pays off. Test R² is slightly higher than validation R² here, which again reflects the small validation set rather than a better-than-expected model; the cross-validated R² of 0.36 is the more conservative estimate.

## 20. Trainable parameters

In [ ]:
pd.DataFrame(
    {
        "trainable parameters": [ModelInspector.parameter_count(m) for m in part3.models.values()],
        "detail": [
            f"{len(ModelInspector.parameter_breakdown(m))} coefficients incl. intercept"
            if "poly" in name or "linear" in name
            else ModelInspector.parameter_breakdown(m)
            for name, m in part3.models.items()
        ],
    },
    index=part3.models.keys(),
)

In [ ]:
terms = part3.models["poly_degree_2"].named_steps["poly"].get_feature_names_out()
print(f"{len(terms)} terms, for example: {', '.join(terms[:4])} ... {', '.join(terms[-4:])}")

- **Polynomial degree 2: 45.** With 8 features, `get_feature_names_out()` produces 8 linear terms, 8 squares and 28 pairwise interactions (44 terms), plus one intercept.
- **Polynomial degree 3: 165.** The 164 terms include every product of up to three features. That is one coefficient for every two training patients, which explains the overfitting.
- **Decision trees: number of nodes.** Each internal node learns a split threshold and each leaf learns a predicted value. The depth-3 tree has 7 thresholds and 8 leaf values (15). The depth-5 tree has 25 thresholds and 26 leaves (51). Which feature to split on is also chosen during training, but it is a discrete choice, not a numeric weight.
- **kNN: 0.** kNN does not learn anything during `fit`. It stores the 331 training rows and does all the work at prediction time by finding the nearest neighbours. Its behaviour is set by hyperparameters (*k*, the distance metric) rather than learned weights, which is the "lazy learner" idea from the kNN workshop.
- **Linear reference: 9.** One weight per feature plus an intercept.

## 21. Logistic regression as a high-risk classifier

Logistic regression cannot predict a progression score; it predicts the probability of a class. Following the framing in section 2, the target becomes **high risk = progression ≥ 214** (the 75th percentile of the training targets). The cut-off is computed from training data only so the validation and test sets do not influence it.

About 25% of training patients are high risk, so this is an imbalanced problem. The two models differ in how they handle that:

- `logistic_default`: standard L2 regularisation (`C = 1.0`), no reweighting.
- `logistic_balanced`: stronger regularisation (`C = 0.1`) and `class_weight="balanced"`, which makes each missed high-risk patient cost about three times as much during training.

R², MAE and MAPE do not apply to class labels, so these models are evaluated with the classification metrics from the Performance Metrics and Log-Loss workshops.

In [ ]:
classifier = ClassificationEvaluator(multi_splits, threshold)
classifier.fit(
    {
        "logistic_default": ModelFactory.logistic(C=1.0),
        "logistic_balanced": ModelFactory.logistic(C=0.1, class_weight="balanced"),
    }
)
classifier.comparison_table()

- Both models rank patients almost equally well (validation ROC-AUC 0.93 and 0.94), so the underlying signal is the same.
- The difference is where they draw the line. The default model catches only 3 of the 7 high-risk validation patients (recall 0.43). The balanced model catches 6 of 7 (recall 0.86) at the cost of more false alarms (precision 0.55).
- The default model has the lower log-loss because it is more confident on the majority class. That is a good reminder from the Log-Loss workshop that the metric being optimised is not always the metric that matches the use case.

For screening, recall wins. **Chosen classifier: `logistic_balanced`.**

In [ ]:
best_classifier = "logistic_balanced"
classifier.test_report(best_classifier)

In [ ]:
viz.confusion_matrix(
    classifier.confusion(best_classifier, "test"),
    "Balanced logistic regression on the test set", "logistic_confusion_matrix",
)

In [ ]:
viz.importance(
    ModelInspector.feature_importance(classifier.models[best_classifier], selected),
    "Logistic regression weights (standardised features)", "logistic_weights",
)

On the test set the balanced model catches **14 of 19 high-risk patients (recall 0.74)** with 7 false alarms among 48 lower-risk patients, and keeps a ROC-AUC of 0.88. BMI, blood pressure and `ltg` push risk up; HDL pushes it down. The weights agree with the EDA, which is reassuring.

**Regression vs classification for screening.** The same 214 cut-off can be applied to the regression models' predicted scores:

In [ ]:
part3.screening_table(threshold, split='test')

Every regression model misses at least 8 of the 19 high-risk test patients, and the chosen kNN misses 13. kNN averages 11 neighbours, which pulls extreme patients toward the middle, so its predictions rarely cross the cut-off. The balanced logistic model misses 5. When the question is "should this patient be flagged?", modelling that question directly works far better than thresholding a score.

## 22. Part 3 conclusions

**Performance summary**

| Task | Model | Test result |
|---|---|---|
| Predict the score | kNN, k = 11, 8 features | R² 0.49, MAE 42.8, MAPE 36.3% |
| Flag high risk | Logistic, balanced, 8 features | recall 0.74, precision 0.67, ROC-AUC 0.88 |
| Reference | BMI-only line (Part 2) | R² 0.28, MAE 51.5, MAPE 43.2% |

**Where the kNN model fails**

In [ ]:
knn_test_X, knn_test_y = multi_splits["test"]
viz.actual_vs_predicted(
    knn_test_y, knn_model.predict(knn_test_X), "kNN (k = 11) on the test set: actual vs predicted",
    "part3_actual_vs_predicted", threshold=threshold,
)

In [ ]:
part3_bands = part3.error_by_range(best_multi, split="test")
viz.error_by_band(part3_bands, "kNN errors by actual progression (test set)", "part3_error_bands")
part3_bands

In [ ]:
viz.importance(
    ModelInspector.feature_importance(part3.models["tree_depth_5"], selected),
    "What the depth-5 tree relies on", "tree_importance",
)

- The middle of the range is now predicted well (MAE 23 for scores between 92 and 150, down from 39 with BMI alone).
- **The extremes remain the weak spot.** The bottom quarter is still over-predicted (bias +50) and the top quarter under-predicted (bias -54). Averaging neighbours can never produce a value more extreme than the patients it averages.
- **As a screener the kNN regressor is barely better than the BMI line** (recall 0.32 vs 0.26, far below the logistic model's 0.74). Better average error did not translate into catching more high-risk patients.
- The tree's feature importances show it relies on BMI for over 60% of its decisions, so it is close to a BMI model with a few extra splits, which explains why it did not beat kNN.

**Limitations**

- kNN cannot extrapolate. A patient with values beyond the training range is matched to the nearest edge patients and receives an "average edge" prediction. The dashboard warns when an input falls outside the training range for this reason.
- kNN stores every training patient, so deploying it means shipping the patient data with the model, a privacy concern in a clinical setting.
- The high-risk cut-off (top 25%) is a modelling choice, not a clinical definition. A physician should set it based on what level of progression warrants intervention.
- The validation set is small enough that model rankings within a few MAE points should be read as ties.
- 442 patients from one study is a small and possibly unrepresentative sample.

---

# Overall conclusions

1. **More features help.** Going from BMI alone to eight features raises test R² from 0.28 to 0.49 and cuts MAE from 51.5 to 42.8.
2. **More flexibility does not.** The degree-3 polynomial and the depth-5 tree overfit, and the plain linear model was as good as anything in cross-validation. With 442 patients, simple models are the safer choice.
3. **The validation set alone can mislead.** It favoured a degree-5 polynomial that cross-validation and common sense both rejected.
4. **Every regression model under-predicts the fastest progressors.** That is a structural consequence of minimising average error on a target with a long upper tail.
5. **For screening, classify directly.** The balanced logistic regression catches 74% of high-risk test patients against 32% for the best regressor.

**Recommendation.** Use the balanced logistic regression to flag patients and the kNN score as supporting context for how far progression might go. With more time, I would test a regularised linear model (ridge or lasso), tune the classification threshold to a target recall agreed with clinicians, and validate on an independent patient group.

**References**

- Efron, B., Hastie, T., Johnstone, I. and Tibshirani, R. (2004). *Least Angle Regression*. Annals of Statistics, 32(2), 407 to 499.
- scikit-learn: [Diabetes dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) and [`load_diabetes`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html).
- Course workshops: Performance Metrics Classification, K-Nearest Neighbors, Logistic Regression Classifier (Log-Loss).

---

# Screening dashboard

The chosen models are saved to `models/` with the metadata the dashboard needs (feature ranges, the risk cut-off and test scores). The app lets a user enter a patient's baseline values and returns the predicted progression score and the probability of being high risk.

In [ ]:
registry = ModelRegistry("models")
registry.save(knn_model, "knn_regressor")
registry.save(part3.models["linear_reference"], "linear_regressor")
registry.save(bmi_model, "bmi_regressor")
registry.save(classifier.models[best_classifier], "risk_classifier")

train_X, train_y = multi_splits["train"]
registry.write_metadata(
    {
        "features": selected,
        "descriptions": {f: data.DESCRIPTIONS[f] for f in selected},
        "ranges": {
            f: {"min": float(train_X[f].min()), "max": float(train_X[f].max()), "median": float(train_X[f].median())}
            for f in selected
        },
        "risk_threshold": threshold,
        "training_targets": train_y.round(1).tolist(),
        "test_scores": {
            "knn_regressor": part3.scores(knn_model, "test"),
            "linear_regressor": part3.scores(part3.models["linear_reference"], "test"),
            "bmi_regressor": part2.scores(bmi_model, "test"),
            "risk_classifier": classifier.scores(classifier.models[best_classifier], "test"),
        },
    }
)
sorted(p.name for p in registry.directory.iterdir())

Launch the dashboard from the project folder:

```bash
streamlit run app.py
```

Or run the next cell to start it from the notebook (stop it with the interrupt button).

In [ ]:
# !streamlit run app.py